# Сравнение со sklearn


In [3]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, SGDRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.model_selection import KFold, LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

root = Path.cwd()

In [4]:
# Берём только ячейки с собственными реализациями из первого ноутбука.
notebook = json.loads((root / "02_lab_implementation.ipynb").read_text(encoding="utf-8"))
for cell in notebook["cells"]:
    if "implementation" in cell.get("metadata", {}).get("tags", []):
        exec("".join(cell["source"]))

## Подготовка данных

Используем те же признаки, очистку и одинаковые разбиения для собственных методов и sklearn.

In [5]:
raw = pd.read_csv(root / "train.csv")
valid_target = raw.RiskScore.notna() & ~raw.RiskScore.isin([-9999999, 10000000])
data = raw.loc[valid_target].drop_duplicates().dropna().reset_index(drop=True)

features = ["CreditScore", "MonthlyIncome", "DebtToIncomeRatio", "InterestRate",
            "LengthOfCreditHistory", "CreditCardUtilizationRate", "NetWorth", "BankruptcyHistory"]
x = data[features].to_numpy(dtype=float)
y = data.RiskScore.to_numpy(dtype=float)

seed = 42
own_folds = list(my_k_fold(len(y), n_splits=5, shuffle=True, random_state=seed))
sklearn_folds = list(KFold(n_splits=5, shuffle=True, random_state=seed).split(x))
train_idx, valid_idx = own_folds[0]
x_train, x_valid = x[train_idx], x[valid_idx]
y_train, y_valid = y[train_idx], y[valid_idx]
print(f"Обучение: {len(y_train)} строк, проверка: {len(y_valid)} строк")

Обучение: 7465 строк, проверка: 1867 строк


## Нормализация

Сравниваю преобразования обучающей и проверочной выборок. Добавляю
константный столбец для проверки случая с нулевым знаменателем.

In [6]:
# Проверяем нормализацию на одних и тех же данных.
for own_cls, sklearn_cls in [(MyStandardScaler, StandardScaler), (MyMinMaxScaler, MinMaxScaler)]:
    own = own_cls().fit(x_train)
    ref = sklearn_cls().fit(x_train)
    difference = np.max(np.abs(own.transform(x_valid) - ref.transform(x_valid)))
    print(own_cls.__name__, "максимальная разница:", difference)
    assert difference < 1e-10

MyStandardScaler максимальная разница: 0.0
MyMinMaxScaler максимальная разница: 1.1102230246251565e-16


## Сравнение аналитического метода, GD и SGD

У всех методов одинаковые обучающие и проверочные данные. Сравниваем значения метрик; SGD может отличаться из-за случайного порядка объектов и особенностей оптимизатора sklearn.

In [7]:
# Все модели обучаем на одинаково нормализованных признаках.
scaler = MyStandardScaler().fit(x_train)
z_train = scaler.transform(x_train)
z_valid = scaler.transform(x_valid)

models = {
    "sklearn OLS": LinearRegression(),
    "Свой OLS": MyLinearRegression("analytical"),
    "Свой GD": MyLinearRegression("gd", learning_rate=0.08, epochs=2500),
    "Свой SGD": MyLinearRegression("sgd", learning_rate=0.05, epochs=25),
    "sklearn SGD": SGDRegressor(penalty=None, learning_rate="invscaling", eta0=0.05,
                                power_t=0.5, max_iter=25, tol=None, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(z_train, y_train)
    pred = model.predict(z_valid)
    results.append([name, my_mse(y_valid, pred), my_mae(y_valid, pred), my_r2(y_valid, pred)])
display(pd.DataFrame(results, columns=["Модель", "MSE", "MAE", "R²"]).set_index("Модель"))

,MSE,MAE,R²
Модель,,,
sklearn OLS,51.327400,5.742664,0.828592
Свой OLS,51.327400,5.742664,0.828592
Свой GD,51.327400,5.742664,0.828592
Свой SGD,51.335519,5.749207,0.828565
sklearn SGD,51.323548,5.742567,0.828605


## Сравнение метрик

Сравниваем MSE, MAE, R² и MAPE на одинаковых истинных значениях и предсказаниях.

In [8]:
# Одни и те же предсказания для своих метрик и sklearn.
ref_pred = LinearRegression().fit(z_train, y_train).predict(z_valid)
metrics = [
    ("MSE", my_mse, mean_squared_error),
    ("MAE", my_mae, mean_absolute_error),
    ("R²", my_r2, r2_score),
    ("MAPE", my_mape, mean_absolute_percentage_error),
]
for name, own, ref in metrics:
    a, b = own(y_valid, ref_pred), ref(y_valid, ref_pred)
    print(f"{name}: своя = {a:.6f}, sklearn = {b:.6f}")

MSE: своя = 51.327400, sklearn = 51.327400
MAE: своя = 5.742664, sklearn = 5.742664
R²: своя = 0.828592, sklearn = 0.828592
MAPE: своя = 0.134385, sklearn = 0.134385


## Сравнение кросс-валидации

Сравниваем индексы train/validation, затем out-of-fold предсказания. Для LOO берём 200 объектов, чтобы не обучать модель тысячи раз.

In [9]:
# Сначала проверяем, что разбиения совпадают со sklearn.
for name, sample_x, sample_y, own_splits, sklearn_splits in [
    ("K-fold", x, y, list(my_k_fold(len(y), 5)), list(KFold(5, shuffle=True, random_state=42).split(x))),
    ("LOO", x[:200], y[:200], list(my_leave_one_out(200)), list(LeaveOneOut().split(x[:200]))),
]:
    for (train1, valid1), (train2, valid2) in zip(own_splits, sklearn_splits):
        np.testing.assert_array_equal(train1, train2)
        np.testing.assert_array_equal(valid1, valid2)

    own_pred = my_cross_val_predict(MyLinearRegression, sample_x, sample_y, own_splits, MyStandardScaler)
    ref_pred = cross_val_predict(make_pipeline(StandardScaler(), LinearRegression()),
                                 sample_x, sample_y, cv=sklearn_splits)
    print(name, "MSE:", my_mse(sample_y, own_pred),
          "sklearn:", mean_squared_error(sample_y, ref_pred))

K-fold MSE: 52.97722992494316 sklearn: 52.97722992494316
LOO MSE: 43.51489501749762 sklearn: 43.51489501749759


Собственные реализации сравниваются со sklearn на одинаковых данных. Для аналитического решения и GD ожидаются близкие результаты; SGD может отличаться из-за настроек обучения. Кросс-валидация проверяется по индексам и прогнозам.